In [2]:
# Import required libraries,load dataset, select input features and print dataset statistics.

import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
import os
from sklearn.preprocessing import LabelEncoder

FEATURES_PATH = "../../datasets/processed/features.csv"
MODEL_DIR = "../../ml-service/models"
os.makedirs(MODEL_DIR, exist_ok=True)

df = pd.read_csv(FEATURES_PATH)
df['datetimeLocal'] = pd.to_datetime(df['datetimeLocal'])
df = df.sort_values('datetimeLocal').reset_index(drop=True)

feature_cols = [c for c in df.columns if any(s in c for s in ['_lag', '_roll', '_sin', '_cos', 'station_'])
                or c in ['hour', 'dayofweek', 'month', 'is_weekend']]
print(f"{len(feature_cols)} features, {len(df)} rows")

42 features, 6190 rows


In [3]:
# Training and testing of data

cutoff = df['datetimeLocal'].quantile(0.8)
train = df[df['datetimeLocal'] <= cutoff]
test  = df[df['datetimeLocal']  > cutoff]

print(f"Train: {len(train)} rows (up to {cutoff})")
print(f"Test:  {len(test)} rows (after {cutoff})")

X_train, y_train = train[feature_cols], train['AQI']
X_test,  y_test  = test[feature_cols],  test['AQI']

Train: 4952 rows (up to 2022-04-25 00:12:00+05:30)
Test:  1238 rows (after 2022-04-25 00:12:00+05:30)


In [4]:
# Regression

reg = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,)
reg.fit(X_train, y_train)
reg_pred = reg.predict(X_test)
print("Regressor trained.")

Regressor trained.


In [5]:
# Classification of data

le = LabelEncoder()
y_train_c = le.fit_transform(train['AQI_category'])
y_test_c  = le.transform(test['AQI_category'])
clf = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss',
)
clf.fit(X_train, y_train_c)
clf_pred = clf.predict(X_test)
print("Classifier trained. Categories:", list(le.classes_))

Classifier trained. Categories: ['Good', 'Moderate', 'Poor', 'Satisfactory', 'Very Poor']


In [6]:
# Save model

joblib.dump(reg, f"{MODEL_DIR}/aqi_regressor.joblib")
joblib.dump(clf, f"{MODEL_DIR}/aqi_classifier.joblib")
joblib.dump(le,  f"{MODEL_DIR}/category_label_encoder.joblib")
joblib.dump(feature_cols, f"{MODEL_DIR}/feature_columns.joblib")

test_results = test[['location_name', 'datetimeLocal', 'AQI', 'AQI_category']].copy()
test_results['AQI_pred'] = reg_pred
test_results['AQI_category_pred'] = le.inverse_transform(clf_pred)
test_results.to_csv("../data/processed/test_predictions.csv", index=False)

print("Saved models to", MODEL_DIR)
print("Saved test predictions to ../data/processed/test_predictions.csv")

Saved models to ../../ml-service/models
Saved test predictions to ../data/processed/test_predictions.csv
